In [2]:
!pip install -q openenv-core peft trl==0.24.0 accelerate datasets sentence-transformers matplotlib

In [3]:
pip install -q torch==2.10.0+cu128 --index-url https://download.pytorch.org/whl/cu128

Note: you may need to restart the kernel to use updated packages.


In [5]:
!git clone -b kaggle_version https://github.com/Dinesh052/Test.git cn

Cloning into 'cn'...
remote: Enumerating objects: 699, done.
remote: Counting objects: 100% (378/378), done.
remote: Compressing objects: 100% (271/271), done.
remote: Total 699 (delta 179), reused 269 (delta 103), pack-reused 321 (from 1)
Receiving objects: 100% (699/699), 86.77 MiB | 39.90 MiB/s, done.
Resolving deltas: 100% (337/337), done.
error: unable to create file .DS_Store: Permission denied
fatal: unable to checkout working tree
You can inspect what was checked out with 'git status'
and retry with 'git restore --source=HEAD :/'



In [6]:
!cd cn && git checkout -f HEAD

error: unable to create file .DS_Store: Permission denied
Updating files: 100% (90/90), done.
Your branch is up to date with 'origin/kaggle_version'.


In [8]:
!cd cn

In [11]:
!pip install -q mergekit llm-blender weave 2>/dev/null;

import site, os, glob
sp = site.getsitepackages()[0]
# Fix callbacks.py
cb = os.path.join(sp, 'trl', 'trainer', 'callbacks.py')
if os.path.exists(cb):
  src = open(cb).read()
  src = src.replace('import weave', 'weave = None')
  src = src.replace('from weave.trace.context', '# from weave.trace.context')
  src = src.replace('from mergekit', '# from mergekit')
  src = src.replace('import llm_blender', 'llm_blender = None')
  open(cb, 'w').write(src)
# Fix llm_blender TRANSFORMERS_CACHE
for f in glob.glob(os.path.join(sp, 'llm_blender', '**', '*.py'), recursive=True):
  s = open(f).read()
  if 'TRANSFORMERS_CACHE' in s:
      open(f, 'w').write(s.replace('from transformers.utils.hub import TRANSFORMERS_CACHE', 'TRANSFORMERS_CACHE = None'))
print('All patched')

Argument expected for the -c option
usage: python [option] ... [-c cmd | -m mod | file | -] [arg] ...
Try `python -h' for more information.
All patched


In [18]:
!pip install --force-reinstall trl==0.14.0

  Using cached trl-0.14.0-py3-none-any.whl (313 kB)
  Using cached transformers-5.6.2-py3-none-any.whl (10.4 MB)
  Using cached datasets-4.8.4-py3-none-any.whl (526 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)
  Using cached rich-15.0.0-py3-none-any.whl (310 kB)
  Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (507 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 88.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 63.9 MB/s eta 0:00:00
  Using cached huggingface_hub-1.12.0-py3-none-any.whl (646 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 79.5 MB/s eta 0:00:00
  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.8 MB)
  Using cached torch-2.11.0-cp310-cp310-manylinux_2_28_x86_64.whl (530.6 MB)
  Using cached pyarrow-24.0.0-cp310-cp310-manylinux_2_28_x86_64.whl (48.8 MB)
  Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)
  Using ca

In [25]:
!pip install --force-reinstall --no-cache-dir trl==0.14.0 2>&1 | tail -3

import site, os
sp = site.getsitepackages()[0]
f = os.path.join(sp, 'trl', 'trainer', 'grpo_trainer.py')
lines = open(f).readlines()
out = []
for line in lines:
  if 'from vllm import' in line:
      indent = line[:len(line) - len(line.lstrip())]
      out.append(indent + 'try:\n')
      out.append(indent + '    from vllm import LLM, SamplingParams\n')
      out.append(indent + 'except ImportError:\n')
      out.append(indent + '    LLM = None\n')
      out.append(indent + '    SamplingParams = None\n')
  else:
      out.append(line)
open(f, 'w').writelines(out)
print('Patched grpo_trainer.py')


mcp 1.27.0 requires pydantic<3.0.0,>=2.11.0, but you have pydantic 2.10.6 which is incompatible.
fastmcp 3.2.4 requires pydantic[email]>=2.11.7, but you have pydantic 2.10.6 which is incompatible.
Patched grpo_trainer.py


In [29]:
# Patch GRPOTrainer.__init__ to add warnings_issued before it's accessed
import site, os
sp = site.getsitepackages()[0]
f = os.path.join(sp, 'trl', 'trainer', 'grpo_trainer.py')
src = open(f).read()
src = src.replace(
  'model.warnings_issued[\"estimate_tokens\"] = True',
  'if not hasattr(model, \"warnings_issued\"): model.warnings_issued = {}\n        model.warnings_issued[\"estimate_tokens\"] = True'
)
open(f, 'w').write(src)
print('Patched warnings_issued')

Patched warnings_issued


In [30]:
!cd cn && sed -i 's/loss_type="dr_grpo",.*$/# loss_type removed for trl 0.14 compat/' training/train_local_v2.py && python training/train_local_v2.py --num-episodes 512 --num-epochs 3 --model Qwen/Qwen2.5-7B-Instruct --lora-r 32 2>&1 | tee train.log

[gpu] NVIDIA A100-SXM4-80GB | 79.2 GB
[model] Unsloth unavailable (No module named 'unsloth'). Using transformers + LoRA.
[model] Loading Qwen/Qwen2.5-7B-Instruct (bf16, no quant)...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 339/339 [00:03<00:00, 99.00it/s] 
trainable params: 20,185,088 || all params: 7,635,801,600 || trainable%: 0.2643
[model] Loaded (trl)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7158.58it/s]
[data] 512 prompts, phase distribution: {'opening': 172, 'negotiation': 296, 'resolution': 42, 'other': 0, 'terminal': 2}
[data] Difficulty distribution: {'easy': 137, 'medium': 137, 'hard': 238}
[train] Starting GRPO v2...
  Prompts: 512, Generations: 4, Epochs: 3, Multi-turn: 4 steps
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Update

In [32]:
!pip install trl==0.24.0

import site, os, glob
sp = site.getsitepackages()[0]
# Patch grpo_trainer vllm
f = os.path.join(sp, 'trl', 'trainer', 'grpo_trainer.py')
lines = open(f).readlines()
out = []
for line in lines:
  if 'from vllm import' in line:
      indent = line[:len(line) - len(line.lstrip())]
      out.append(indent + 'try:\n')
      out.append(indent + '    from vllm import LLM, SamplingParams\n')
      out.append(indent + 'except ImportError:\n')
      out.append(indent + '    LLM = None\n')
      out.append(indent + '    SamplingParams = None\n')
  else:
      out.append(line)
open(f, 'w').writelines(out)
# Patch callbacks
cb = os.path.join(sp, 'trl', 'trainer', 'callbacks.py')
src = open(cb).read()
src = src.replace('import weave', 'weave = None')
src = src.replace('from weave.trace.context', '# from weave.trace.context')
src = src.replace('import llm_blender', 'llm_blender = None')
open(cb, 'w').write(src)
# Patch llm_blender
for f2 in glob.glob(os.path.join(sp, 'llm_blender', '**', '*.py'), recursive=True):
  s = open(f2).read()
  if 'TRANSFORMERS_CACHE' in s:
      open(f2, 'w').write(s.replace('from transformers.utils.hub import TRANSFORMERS_CACHE', 'TRANSFORMERS_CACHE = None'))
print('All patched')

All patched


In [34]:
!cd cn && sed -i '/loss_type/d' training/train_local_v2.py && python training/train_local_v2.py --num-episodes 512 --num-epochs 3 --model Qwen/Qwen2.5-7B-Instruct --lora-r 32 2>&1 | tee train.log

[gpu] NVIDIA A100-SXM4-80GB | 79.2 GB
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
[model] Loading Qwen/Qwen2.5-7B-Instruct via Unsloth...
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.25 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading weights: 100%|██████████| 339/339 [00:03<00:00, 95.66it/s] 
unsloth/Qwen2.5-7B-Instruct does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Not an error, but 

In [35]:
!cd cn && pip install -q matplotlib 


In [36]:
!cd cn && \
python eval/checkpoint_league.py --root ./crisis-negotiator-trained-v2 --n 20 2>&1 | tee checkpoint_league.log && \
python eval/eval_baselines.py --n 50 --difficulties easy,medium,hard --include-trained && \
python eval/plot_belief_convergence.py --n 50 && \
python eval/reward_gaming_audit.py --n 10 --out results/reward_gaming_audit.json && \
python eval/long_horizon_benchmark.py --n 12 --out results/long_horizon_benchmark.json && \
python eval/ablation_mini_table.py --n 18 --out results/ablation_mini_table.json

Found 2 checkpoints under crisis-negotiator-trained-v2

Evaluating checkpoint-736 ...
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 434/434 [00:01<00:00, 305.05it/s]
Traceback (most recent call last):
  File "/data/cn/eval/checkpoint_league.py", line 133, in <module>
    main()
  File "/data/cn/eval/checkpoint_league.py", line 82, in main
    row = evaluate_checkpoint(
  File "/data/cn/eval/checkpoint_league.py", line 45, in evaluate_checkpoint
    policy = TrainedPolicy(adapter_dir=str(ckpt_dir), base_model=base_model)
  File "/data/cn/eval/eval_baselines.py", line 179, in __init__
    self.model = PeftModel.from_pretrained(base, adapter_dir)
  File "/home/user/miniconda/lib/python3.10/site-packages/peft/peft_model.py", line 582, in from_pretrained
    load_result = model.load_adapter(
  File "/home/user/miniconda/lib/python

In [ ]:
!HF_TOKEN=hf_REDACTED hf upload dinhacks/crisis /data/export_final.zip export_final.zip --repo-type space